<a href="https://colab.research.google.com/github/vishnuvs0712-maker/Machine-Learning-codes/blob/main/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade gymnasium stable-baselines3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from collections import deque
import heapq
import random
import time

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO

from IPython.display import HTML, display

print("Libraries loaded successfully!")

In [ ]:
ACTIONS = {
    0: (-1, 0),   # UP
    1: (1, 0),    # DOWN
    2: (0, -1),   # LEFT
    3: (0, 1)     # RIGHT
}

ACTION_NAMES = {
    0: "UP ↑",
    1: "DOWN ↓",
    2: "LEFT ←",
    3: "RIGHT →"
}

print(ACTION_NAMES)

In [ ]:
def check_reachable(maze, start, goal):

    queue = deque([start])
    visited = {start}

    while queue:

        r, c = queue.popleft()

        if (r, c) == goal:
            return True

        for dr, dc in ACTIONS.values():

            nr = r + dr
            nc = c + dc

            if (
                0 <= nr < maze.shape[0]
                and 0 <= nc < maze.shape[1]
                and maze[nr, nc] == 0
                and (nr, nc) not in visited
            ):

                visited.add((nr, nc))
                queue.append((nr, nc))

    return False


def generate_solvable_maze(
    size,
    wall_probability=0.30,
    seed=42
):

    rng = np.random.default_rng(seed)

    start = (0, 0)
    goal = (size - 1, size - 1)

    attempts = 0

    while True:

        attempts += 1

        # 0 = free cell
        # 1 = wall

        maze = (
            rng.random((size, size))
            < wall_probability
        ).astype(np.int32)

        # Start and goal must be free
        maze[start] = 0
        maze[goal] = 0

        # Check if path exists
        if check_reachable(
            maze,
            start,
            goal
        ):

            print(
                f"{size}×{size} maze generated "
                f"after {attempts} attempt(s)."
            )

            return maze

In [ ]:
maze_8 = generate_solvable_maze(
    size=8,
    wall_probability=0.30,
    seed=42
)

maze_16 = generate_solvable_maze(
    size=16,
    wall_probability=0.30,
    seed=123
)

In [ ]:
def display_maze(
    maze,
    title="Maze"
):

    n = maze.shape[0]

    fig, ax = plt.subplots(
        figsize=(7, 7)
    )

    ax.imshow(
        maze,
        cmap="binary",
        vmin=0,
        vmax=1
    )

    # Start
    ax.scatter(
        0,
        0,
        s=300,
        c="blue",
        marker="o",
        edgecolors="black",
        linewidths=2,
        label="Start"
    )

    # Goal
    ax.scatter(
        n - 1,
        n - 1,
        s=300,
        c="green",
        marker="*",
        edgecolors="black",
        linewidths=2,
        label="Goal"
    )

    ax.set_title(
        title,
        fontsize=16
    )

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))

    ax.grid(
        True,
        linewidth=0.8
    )

    ax.legend()

    plt.show()


display_maze(
    maze_8,
    "8 × 8 Maze"
)

display_maze(
    maze_16,
    "16 × 16 Maze"
)

In [ ]:
print(
    "8×8 reachable:",
    check_reachable(
        maze_8,
        (0, 0),
        (7, 7)
    )
)

print(
    "16×16 reachable:",
    check_reachable(
        maze_16,
        (0, 0),
        (15, 15)
    )
)

In [ ]:
def get_neighbors(position, maze):

    r, c = position

    neighbors = []

    for action, (dr, dc) in ACTIONS.items():

        nr = r + dr
        nc = c + dc

        if (
            0 <= nr < maze.shape[0]
            and 0 <= nc < maze.shape[1]
            and maze[nr, nc] == 0
        ):

            neighbors.append(
                ((nr, nc), action)
            )

    return neighbors


def reconstruct_path(
    parent,
    start,
    goal
):

    if goal not in parent:
        return []

    path = []

    current = goal

    while current is not None:

        path.append(current)

        if current == start:
            break

        current = parent[current]

    path.reverse()

    return path

In [ ]:
def bfs(
    maze,
    start,
    goal
):

    start_time = time.perf_counter()

    queue = deque([start])

    visited = {start}

    parent = {
        start: None
    }

    explored = []

    while queue:

        current = queue.popleft()

        explored.append(current)

        if current == goal:
            break

        for neighbor, action in get_neighbors(
            current,
            maze
        ):

            if neighbor not in visited:

                visited.add(neighbor)

                parent[neighbor] = current

                queue.append(neighbor)

    path = reconstruct_path(
        parent,
        start,
        goal
    )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    return {
        "path": path,
        "explored": explored,
        "time": elapsed
    }

In [ ]:
def dfs(
    maze,
    start,
    goal
):

    start_time = time.perf_counter()

    stack = [start]

    visited = {start}

    parent = {
        start: None
    }

    explored = []

    while stack:

        current = stack.pop()

        explored.append(current)

        if current == goal:
            break

        neighbors = get_neighbors(
            current,
            maze
        )

        for neighbor, action in reversed(
            neighbors
        ):

            if neighbor not in visited:

                visited.add(neighbor)

                parent[neighbor] = current

                stack.append(neighbor)

    path = reconstruct_path(
        parent,
        start,
        goal
    )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    return {
        "path": path,
        "explored": explored,
        "time": elapsed
    }

In [ ]:
def manhattan_distance(
    a,
    b
):

    return (
        abs(a[0] - b[0])
        +
        abs(a[1] - b[1])
    )


def best_first_search(
    maze,
    start,
    goal
):

    start_time = time.perf_counter()

    heap = []

    counter = 0

    heapq.heappush(
        heap,
        (
            manhattan_distance(
                start,
                goal
            ),
            counter,
            start
        )
    )

    visited = set()

    parent = {
        start: None
    }

    explored = []

    while heap:

        _, _, current = heapq.heappop(
            heap
        )

        if current in visited:
            continue

        visited.add(current)

        explored.append(current)

        if current == goal:
            break

        for neighbor, action in get_neighbors(
            current,
            maze
        ):

            if neighbor not in visited:

                counter += 1

                parent[neighbor] = current

                priority = manhattan_distance(
                    neighbor,
                    goal
                )

                heapq.heappush(
                    heap,
                    (
                        priority,
                        counter,
                        neighbor
                    )
                )

    path = reconstruct_path(
        parent,
        start,
        goal
    )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    return {
        "path": path,
        "explored": explored,
        "time": elapsed
    }

In [ ]:
def a_star(
    maze,
    start,
    goal
):

    start_time = time.perf_counter()

    heap = []

    counter = 0

    g_cost = {
        start: 0
    }

    parent = {
        start: None
    }

    heapq.heappush(
        heap,
        (
            manhattan_distance(
                start,
                goal
            ),
            counter,
            start
        )
    )

    visited = set()

    explored = []

    while heap:

        _, _, current = heapq.heappop(
            heap
        )

        if current in visited:
            continue

        visited.add(current)

        explored.append(current)

        if current == goal:
            break

        for neighbor, action in get_neighbors(
            current,
            maze
        ):

            new_g = (
                g_cost[current] + 1
            )

            if (
                neighbor not in g_cost
                or
                new_g < g_cost[neighbor]
            ):

                g_cost[neighbor] = new_g

                parent[neighbor] = current

                h = manhattan_distance(
                    neighbor,
                    goal
                )

                f = new_g + h

                counter += 1

                heapq.heappush(
                    heap,
                    (
                        f,
                        counter,
                        neighbor
                    )
                )

    path = reconstruct_path(
        parent,
        start,
        goal
    )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    return {
        "path": path,
        "explored": explored,
        "time": elapsed
    }

In [ ]:
def run_all_search_algorithms(
    maze
):

    start = (0, 0)

    goal = (
        maze.shape[0] - 1,
        maze.shape[1] - 1
    )

    results = {}

    results["BFS"] = bfs(
        maze,
        start,
        goal
    )

    results["DFS"] = dfs(
        maze,
        start,
        goal
    )

    results["Best First"] = best_first_search(
        maze,
        start,
        goal
    )

    results["A*"] = a_star(
        maze,
        start,
        goal
    )

    return results


results_8 = run_all_search_algorithms(
    maze_8
)

results_16 = run_all_search_algorithms(
    maze_16
)

print("Search algorithms completed!")

In [ ]:
def print_results(
    results,
    size
):

    print("=" * 70)

    print(
        f"{size}×{size} MAZE RESULTS"
    )

    print("=" * 70)

    print(
        f"{'Algorithm':<18}"
        f"{'Path Length':<15}"
        f"{'Explored':<15}"
        f"{'Time (sec)':<15}"
    )

    print("-" * 70)

    for name, result in results.items():

        print(
            f"{name:<18}"
            f"{len(result['path']):<15}"
            f"{len(result['explored']):<15}"
            f"{result['time']:<15.6f}"
        )


print_results(
    results_8,
    8
)

print_results(
    results_16,
    16
)

In [ ]:
def animate_search(
    maze,
    result,
    algorithm_name,
    interval=250
):

    exploration = result["explored"]
    path = result["path"]

    n = maze.shape[0]

    # Number of frames
    total_frames = (
        len(exploration)
        +
        len(path)
        +
        5
    )

    fig, ax = plt.subplots(
        figsize=(8, 8)
    )

    def draw(frame):

        ax.clear()

        # Maze
        ax.imshow(
            maze,
            cmap="binary",
            vmin=0,
            vmax=1
        )

        # Start
        ax.scatter(
            0,
            0,
            s=300,
            c="blue",
            marker="o",
            edgecolors="black",
            linewidths=2
        )

        # Goal
        ax.scatter(
            n - 1,
            n - 1,
            s=300,
            c="green",
            marker="*",
            edgecolors="black",
            linewidths=2
        )

        # Grid
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))

        ax.grid(
            True,
            linewidth=0.8
        )

        # ----------------------------------
        # Exploration phase
        # ----------------------------------

        exp_count = min(
            frame,
            len(exploration)
        )

        current_explored = (
            exploration[:exp_count]
        )

        if current_explored:

            rows = [
                p[0]
                for p in current_explored
            ]

            cols = [
                p[1]
                for p in current_explored
            ]

            ax.scatter(
                cols,
                rows,
                s=70,
                c="gold",
                marker="s",
                alpha=0.7
            )

        # ----------------------------------
        # Path phase
        # ----------------------------------

        if frame >= len(exploration):

            path_frame = (
                frame
                - len(exploration)
            )

            path_count = min(
                path_frame + 1,
                len(path)
            )

            current_path = (
                path[:path_count]
            )

            if current_path:

                rows = [
                    p[0]
                    for p in current_path
                ]

                cols = [
                    p[1]
                    for p in current_path
                ]

                # Path
                ax.plot(
                    cols,
                    rows,
                    linewidth=4,
                    c="purple"
                )

                # Agent
                agent = current_path[-1]

                ax.scatter(
                    agent[1],
                    agent[0],
                    s=400,
                    c="dodgerblue",
                    marker="o",
                    edgecolors="black",
                    linewidths=3
                )

                # Next movement
                if (
                    path_count
                    < len(path)
                ):

                    next_cell = (
                        path[path_count]
                    )

                    dr = (
                        next_cell[0]
                        - agent[0]
                    )

                    dc = (
                        next_cell[1]
                        - agent[1]
                    )

                    if dr == -1:
                        direction = "UP ↑"

                    elif dr == 1:
                        direction = "DOWN ↓"

                    elif dc == -1:
                        direction = "LEFT ←"

                    else:
                        direction = "RIGHT →"

                    ax.annotate(
                        "",
                        xy=(
                            next_cell[1],
                            next_cell[0]
                        ),
                        xytext=(
                            agent[1],
                            agent[0]
                        ),
                        arrowprops=dict(
                            arrowstyle="->",
                            lw=3,
                            color="red"
                        )
                    )

                    ax.set_title(
                        f"{algorithm_name}\n"
                        f"Agent: {agent}   "
                        f"Next move: {direction}",
                        fontsize=14
                    )

                else:

                    ax.set_title(
                        f"{algorithm_name}\n"
                        f"GOAL REACHED ✓",
                        fontsize=16
                    )

        else:

            if exploration:

                current = exploration[
                    min(
                        frame,
                        len(exploration)-1
                    )
                ]

                ax.scatter(
                    current[1],
                    current[0],
                    s=350,
                    c="dodgerblue",
                    marker="o",
                    edgecolors="black",
                    linewidths=3
                )

                ax.set_title(
                    f"{algorithm_name}\n"
                    f"Exploring: {current}",
                    fontsize=14
                )

        ax.set_xlabel(
            "Agent navigates around obstacles"
        )

    anim = animation.FuncAnimation(
        fig,
        draw,
        frames=total_frames,
        interval=interval,
        repeat=False
    )

    plt.close(fig)

    return HTML(
        anim.to_jshtml()
    )

In [ ]:
display(
    animate_search(
        maze_8,
        results_8["BFS"],
        "BFS — 8×8",
        interval=250
    )
)

In [ ]:
display(
    animate_search(
        maze_8,
        results_8["DFS"],
        "DFS — 8×8",
        interval=250
    )
)

In [ ]:
display(
    animate_search(
        maze_8,
        results_8["Best First"],
        "Best First Search — 8×8",
        interval=250
    )
)

In [ ]:
display(
    animate_search(
        maze_8,
        results_8["A*"],
        "A* — 8×8",
        interval=250
    )
)

In [ ]:
display(
    animate_search(
        maze_16,
        results_16["BFS"],
        "BFS — 16×16",
        interval=120
    )
)

In [ ]:
display(
    animate_search(
        maze_16,
        results_16["DFS"],
        "DFS — 16×16",
        interval=120
    )
)

In [ ]:
display(
    animate_search(
        maze_16,
        results_16["Best First"],
        "Best First Search — 16×16",
        interval=120
    )
)

In [ ]:
display(
    animate_search(
        maze_16,
        results_16["A*"],
        "A* — 16×16",
        interval=120
    )
)

In [ ]:
class MazePPOEnv(gym.Env):

    def __init__(self, maze):

        super().__init__()

        self.maze = maze.astype(
            np.float32
        )

        self.n = maze.shape[0]

        self.start = (0, 0)

        self.goal = (
            self.n - 1,
            self.n - 1
        )

        # Four possible actions
        self.action_space = spaces.Discrete(4)

        # Three channels:
        #
        # Channel 0 → maze
        # Channel 1 → agent
        # Channel 2 → goal

        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(
                3,
                self.n,
                self.n
            ),
            dtype=np.float32
        )

        self.max_steps = (
            self.n * self.n * 4
        )

    def distance_to_goal(
        self,
        position
    ):

        r, c = position

        gr, gc = self.goal

        return (
            abs(r - gr)
            +
            abs(c - gc)
        )

    def get_observation(self):

        observation = np.zeros(
            (
                3,
                self.n,
                self.n
            ),
            dtype=np.float32
        )

        # Maze
        observation[0] = self.maze

        # Agent
        r, c = self.agent

        observation[
            1,
            r,
            c
        ] = 1.0

        # Goal
        gr, gc = self.goal

        observation[
            2,
            gr,
            gc
        ] = 1.0

        return observation

    def reset(
        self,
        seed=None,
        options=None
    ):

        super().reset(
            seed=seed
        )

        self.agent = self.start

        self.steps = 0

        return (
            self.get_observation(),
            {}
        )

    def step(
        self,
        action
    ):

        self.steps += 1

        r, c = self.agent

        dr, dc = ACTIONS[
            int(action)
        ]

        nr = r + dr
        nc = c + dc

        old_distance = (
            self.distance_to_goal(
                self.agent
            )
        )

        # ---------------------------------
        # Check whether movement is valid
        # ---------------------------------

        valid = (
            0 <= nr < self.n
            and
            0 <= nc < self.n
            and
            self.maze[nr, nc] == 0
        )

        if not valid:

            # Wall / outside maze
            reward = -1.0

        else:

            # Move agent
            self.agent = (
                nr,
                nc
            )

            new_distance = (
                self.distance_to_goal(
                    self.agent
                )
            )

            # Reward for getting closer
            distance_reward = (
                old_distance
                - new_distance
            )

            reward = (
                0.8 * distance_reward
            )

            # Small step penalty
            reward -= 0.05

        # ---------------------------------
        # Goal
        # ---------------------------------

        terminated = (
            self.agent == self.goal
        )

        # ---------------------------------
        # Maximum steps
        # ---------------------------------

        truncated = (
            self.steps >= self.max_steps
        )

        if terminated:

            reward += 30.0

        return (
            self.get_observation(),
            float(reward),
            terminated,
            truncated,
            {}
        )

In [ ]:
env_8 = MazePPOEnv(
    maze_8
)

obs, info = env_8.reset()

print("PPO environment created")
print("-------------------------")
print("Maze size:", maze_8.shape)
print("Agent:", env_8.agent)
print("Goal:", env_8.goal)
print("Observation:", obs.shape)
print("Action space:", env_8.action_space)

In [ ]:
print("Testing PPO environment")
print("=" * 60)

for action in range(4):

    env_8.reset()

    obs, reward, terminated, truncated, info = (
        env_8.step(action)
    )

    print(
        f"Action {action} "
        f"({ACTION_NAMES[action]:8s}) → "
        f"Position {env_8.agent} | "
        f"Reward {reward:.2f}"
    )

In [ ]:
model_8 = PPO(
    "MlpPolicy",
    env_8,

    learning_rate=0.0003,

    n_steps=256,

    batch_size=64,

    gamma=0.99,

    gae_lambda=0.95,

    ent_coef=0.02,

    verbose=1,

    seed=42
)

model_8.learn(
    total_timesteps=100000
)

In [ ]:
def run_ppo_agent(
    model,
    env
):

    obs, info = env.reset()

    path = [
        env.agent
    ]

    actions = []

    rewards = []

    for step in range(
        env.max_steps
    ):

        action, _ = model.predict(
            obs,
            deterministic=True
        )

        action = int(action)

        actions.append(action)

        obs, reward, terminated, truncated, info = (
            env.step(action)
        )

        rewards.append(
            reward
        )

        path.append(
            env.agent
        )

        if (
            terminated
            or
            truncated
        ):
            break

    return (
        path,
        actions,
        rewards,
        terminated
    )


ppo_path_8, ppo_actions_8, ppo_rewards_8, reached_goal_8 = (
    run_ppo_agent(
        model_8,
        env_8
    )
)

print("=" * 60)
print("PPO 8×8 RESULTS")
print("=" * 60)

print(
    "Steps:",
    len(ppo_actions_8)
)

print(
    "Path length:",
    len(ppo_path_8)
)

print(
    "Total reward:",
    round(
        sum(ppo_rewards_8),
        2
    )
)

print(
    "Goal reached:",
    reached_goal_8
)

print(
    "Final position:",
    ppo_path_8[-1]
)

print(
    "Goal:",
    env_8.goal
)

In [ ]:
print("=" * 75)
print("PPO AGENT DECISIONS")
print("=" * 75)

for i in range(
    len(ppo_actions_8)
):

    current = ppo_path_8[i]

    next_position = (
        ppo_path_8[i + 1]
    )

    action = ppo_actions_8[i]

    reward = ppo_rewards_8[i]

    print(
        f"Step {i+1:3d} | "
        f"{current} → {next_position} | "
        f"Action: {ACTION_NAMES[action]:8s} | "
        f"Reward: {reward:7.2f}"
    )

In [ ]:
def animate_ppo(
    maze,
    path,
    actions,
    rewards,
    title,
    interval=400
):

    n = maze.shape[0]

    fig, ax = plt.subplots(
        figsize=(8, 8)
    )

    def draw(frame):

        ax.clear()

        # Maze
        ax.imshow(
            maze,
            cmap="binary",
            vmin=0,
            vmax=1
        )

        # Grid
        ax.set_xticks(
            range(n)
        )

        ax.set_yticks(
            range(n)
        )

        ax.grid(
            True,
            linewidth=0.8
        )

        # Start
        ax.scatter(
            0,
            0,
            s=300,
            c="blue",
            marker="s",
            edgecolors="black",
            linewidths=2
        )

        # Goal
        ax.scatter(
            n - 1,
            n - 1,
            s=300,
            c="green",
            marker="*",
            edgecolors="black",
            linewidths=2
        )

        current = path[
            min(
                frame,
                len(path) - 1
            )
        ]

        # Draw path/trail
        trail = path[
            :frame + 1
        ]

        if len(trail) > 1:

            rows = [
                p[0]
                for p in trail
            ]

            cols = [
                p[1]
                for p in trail
            ]

            ax.plot(
                cols,
                rows,
                linewidth=4,
                c="purple"
            )

        # Agent
        ax.scatter(
            current[1],
            current[0],
            s=400,
            c="dodgerblue",
            marker="o",
            edgecolors="black",
            linewidths=3
        )

        # PPO action
        if frame < len(actions):

            action = actions[frame]

            direction = ACTION_NAMES[
                action
            ]

            dr, dc = ACTIONS[
                action
            ]

            next_r = (
                current[0] + dr
            )

            next_c = (
                current[1] + dc
            )

            # Is PPO attempting a blocked move?
            blocked = not (
                0 <= next_r < n
                and
                0 <= next_c < n
                and
                maze[
                    next_r,
                    next_c
                ] == 0
            )

            if blocked:

                decision_text = (
                    f"BLOCKED! → "
                    f"PPO selected {direction}"
                )

            else:

                decision_text = (
                    f"PPO Decision → "
                    f"{direction}"
                )

            # Draw arrow
            if (
                0 <= next_r < n
                and
                0 <= next_c < n
            ):

                ax.annotate(
                    "",
                    xy=(
                        next_c,
                        next_r
                    ),
                    xytext=(
                        current[1],
                        current[0]
                    ),
                    arrowprops=dict(
                        arrowstyle="->",
                        lw=3,
                        color="red"
                    )
                )

            ax.set_title(
                f"{title}\n"
                f"Position: {current} | "
                f"{decision_text}\n"
                f"Reward: {rewards[frame]:.2f}",
                fontsize=13
            )

        else:

            ax.set_title(
                f"{title}\n"
                "GOAL REACHED ✓",
                fontsize=16
            )

    anim = animation.FuncAnimation(
        fig,
        draw,
        frames=len(path),
        interval=interval,
        repeat=False
    )

    plt.close(fig)

    return HTML(
        anim.to_jshtml()
    )

In [ ]:
display(
    animate_ppo(
        maze_8,
        ppo_path_8,
        ppo_actions_8,
        ppo_rewards_8,
        "PPO Reinforcement Learning — 8×8",
        interval=400
    )
)

In [ ]:
env_16 = MazePPOEnv(
    maze_16
)

print(
    "16×16 PPO environment created."
)

print(
    "Observation shape:",
    env_16.observation_space.shape
)

In [ ]:
model_16 = PPO(
    "MlpPolicy",
    env_16,

    learning_rate=0.0003,

    n_steps=512,

    batch_size=64,

    gamma=0.99,

    gae_lambda=0.95,

    ent_coef=0.02,

    verbose=1,

    seed=42
)

model_16.learn(
    total_timesteps=200000
)

In [ ]:
ppo_path_16, ppo_actions_16, ppo_rewards_16, reached_goal_16 = (
    run_ppo_agent(
        model_16,
        env_16
    )
)

print("=" * 60)
print("PPO 16×16 RESULTS")
print("=" * 60)

print(
    "Steps:",
    len(ppo_actions_16)
)

print(
    "Path length:",
    len(ppo_path_16)
)

print(
    "Total reward:",
    round(
        sum(ppo_rewards_16),
        2
    )
)

print(
    "Goal reached:",
    reached_goal_16
)

print(
    "Final position:",
    ppo_path_16[-1]
)

print(
    "Goal:",
    env_16.goal
)

In [ ]:
display(
    animate_ppo(
        maze_16,
        ppo_path_16,
        ppo_actions_16,
        ppo_rewards_16,
        "PPO Reinforcement Learning — 16×16",
        interval=150
    )
)

In [ ]:
def comparison_graph(
    results,
    title
):

    algorithms = list(
        results.keys()
    )

    path_lengths = [
        len(results[a]["path"])
        for a in algorithms
    ]

    explored_nodes = [
        len(results[a]["explored"])
        for a in algorithms
    ]

    execution_times = [
        results[a]["time"]
        for a in algorithms
    ]

    # ----------------------------
    # Path Length
    # ----------------------------

    plt.figure(
        figsize=(9, 5)
    )

    plt.bar(
        algorithms,
        path_lengths
    )

    plt.title(
        f"{title} — Path Length"
    )

    plt.ylabel(
        "Number of cells"
    )

    plt.show()

    # ----------------------------
    # Explored nodes
    # ----------------------------

    plt.figure(
        figsize=(9, 5)
    )

    plt.bar(
        algorithms,
        explored_nodes
    )

    plt.title(
        f"{title} — Nodes Explored"
    )

    plt.ylabel(
        "Number of cells"
    )

    plt.show()

    # ----------------------------
    # Execution time
    # ----------------------------

    plt.figure(
        figsize=(9, 5)
    )

    plt.bar(
        algorithms,
        execution_times
    )

    plt.title(
        f"{title} — Execution Time"
    )

    plt.ylabel(
        "Seconds"
    )

    plt.show()

In [ ]:
comparison_graph(
    results_8,
    "8×8 Maze"
)

In [ ]:
comparison_graph(
    results_16,
    "16×16 Maze"
)

In [ ]:
print("=" * 80)
print("FINAL ALGORITHM COMPARISON")
print("=" * 80)

print(
    f"{'Maze':<10}"
    f"{'Algorithm':<18}"
    f"{'Path':<12}"
    f"{'Explored':<12}"
    f"{'Time':<12}"
)

print("-" * 80)

for maze_size, results in [
    ("8×8", results_8),
    ("16×16", results_16)
]:

    for name, result in results.items():

        print(
            f"{maze_size:<10}"
            f"{name:<18}"
            f"{len(result['path']):<12}"
            f"{len(result['explored']):<12}"
            f"{result['time']:<12.6f}"
        )

# PPO 8×8
print(
    f"{'8×8':<10}"
    f"{'PPO':<18}"
    f"{len(ppo_path_8):<12}"
    f"{'-':<12}"
    f"{'-':<12}"
)

# PPO 16×16
print(
    f"{'16×16':<10}"
    f"{'PPO':<18}"
    f"{len(ppo_path_16):<12}"
    f"{'-':<12}"
    f"{'-':<12}"
)